# 05 — Regular Expressions
### The `re` module, and how `llm_client.py`'s mock heuristics actually work

`llm_client.py`'s mock mode classifies tickets using regex pattern matching
instead of a real model. This notebook builds up regex syntax piece by
piece until you can read (and extend) its `_KEYWORD_MAP` list line by
line.

## 5.1 The absolute basics — `re.search`

`re.search(pattern, text)` looks for `pattern` **anywhere** inside `text`
and returns a match object if found, `None` otherwise. Patterns are just
strings with special characters that mean "match this kind of thing" rather
than a literal character.

In [2]:
import re

text = "i want a refund for my broken headphones"

match = re.search(r"refund", text)
print(match)                 # a Match object if found
print(bool(match))           # True/False -- the common way to use it in an if-statement

no_match = re.search(r"cancel", text)
print(no_match)              # None


<re.Match object; span=(9, 15), match='refund'>
True
None


Note the `r"..."` — a **raw string**. Regex patterns use backslashes a lot
(`\d`, `\s`, `\b`), and a raw string tells Python not to treat those
backslashes as string escape sequences. Get in the habit of always writing
regex patterns as raw strings.

## 5.2 Alternation — `|` means "or"

`_KEYWORD_MAP` in `llm_client.py` is full of patterns like:

```python
r"don'?t recognize|didn'?t make this|unauthorized transaction|not my transaction"
```

The `|` character means "match any one of these alternatives." This single
pattern matches a ticket containing *any* of four different phrasings.

In [3]:
pattern = r"don't recognize|didn't make this|unauthorized transaction"

for text in [
    "I don't recognize this charge",
    "I didn't make this purchase",
    "there's an unauthorized transaction on my card",
    "everything looks fine, thanks",
]:
    print(f"{bool(re.search(pattern, text))}: {text}")


True: I don't recognize this charge
True: I didn't make this purchase
True: there's an unauthorized transaction on my card
False: everything looks fine, thanks


## 5.3 `'?` — optional characters

`don'?t` means: match `don`, then **optionally** a `'` (the `?` makes the
preceding character optional), then `t`. This single pattern catches both
`"don't"` and `"dont"` — handling a common typo/contraction variant without
writing two separate alternatives.

In [4]:
pattern = r"don'?t"

for text in ["I don't recognize this", "I dont recognize this", "I do not recognize this"]:
    print(f"{bool(re.search(pattern, text))}: {text}")


True: I don't recognize this
True: I dont recognize this
False: I do not recognize this


Note the third case: `"I do not recognize this"` does **not** match —
`don'?t` only handles the apostrophe being optional, not a completely
different phrasing (`"do not"` vs `"don't"`). This is exactly the kind of
gap that makes mock mode a stand-in, not a real classifier — a real LLM
would catch `"do not recognize"` as meaning the same thing without anyone
writing a specific pattern for it.

## 5.4 Grouping and repeated alternation inside a group — `(a|b)`

Parentheses group part of a pattern so alternation or repetition applies to
just that part, not the whole pattern. This line from `_KEYWORD_MAP`:

```python
r"not sure (it|this) was me"
```

...matches `"not sure it was me"` **and** `"not sure this was me"`, because
the `|` only applies inside the parentheses, not across the whole string.

In [5]:
pattern = r"not sure (it|this) was me"

for text in ["I'm not sure it was me", "not sure this was me", "not sure that was me"]:
    print(f"{bool(re.search(pattern, text))}: {text}")


True: I'm not sure it was me
True: not sure this was me
False: not sure that was me


## 5.5 `.*` — match anything in between

`.` matches any single character, `*` means "zero or more of the previous
thing" — together `.*` means "any amount of anything." `_KEYWORD_MAP` uses
this to catch a phrase where other words might sit between two key ideas:

```python
r"(double check|make sure).*(charge|transaction)"
```

This matches `"double check"` **or** `"make sure"`, followed by *anything*,
followed by `"charge"` **or** `"transaction"` — catching phrasings like
`"just wanted to double check, is this charge normal"` where the two
important words aren't adjacent.

In [6]:
pattern = r"(double check|make sure).*(charge|transaction)"

for text in [
    "just wanted to double check, is this charge normal",
    "can you make sure that transaction is legit",
    "double check my address please",   # no charge/transaction mentioned -- should NOT match
]:
    print(f"{bool(re.search(pattern, text))}: {text}")


True: just wanted to double check, is this charge normal
True: can you make sure that transaction is legit
False: double check my address please


## 5.6 Case sensitivity

By default, regex is case-sensitive: `"Refund"` and `"refund"` are
different strings as far as `re.search` is concerned. `llm_client.py` works
around this by lowercasing the ticket text *before* running any pattern
(`text = ticket_text.lower()`), rather than making every pattern handle
both cases.

In [ ]:
pattern = r"refund"

print(bool(re.search(pattern, "I want a Refund")))          # False -- case mismatch
print(bool(re.search(pattern, "I want a Refund".lower())))   # True -- lowercased first, same as llm_client.py's approach


## 5.7 Extracting a value — capture groups and `re.search(...).group()`

Everything so far only answered yes/no. `llm_client.py`'s classifier also
*extracts* an order ID or customer ID if one appears in the ticket text:

```python
order_match = re.search(r"ORD\d{3,}", ticket_text, re.IGNORECASE)
...
"extracted_order_id": order_match.group(0).upper() if order_match else None,
```

`\d` means "any digit," `{3,}` means "3 or more of the previous thing" —
together `ORD\d{3,}` matches `"ORD"` followed by 3+ digits. `re.IGNORECASE`
is a flag that makes the whole pattern case-insensitive without needing to
lowercase the text first (useful here specifically because we want to
*extract* the ID, and don't want to have lowercased it away).

In [7]:
pattern = r"ORD\d{3,}"

text = "Can you check my order ord0042, it hasn't shipped"
match = re.search(pattern, text, re.IGNORECASE)
print(match)
print(match.group(0))          # the whole matched substring
print(match.group(0).upper())  # normalized to uppercase, same as the project does


<re.Match object; span=(23, 30), match='ord0042'>
ord0042
ORD0042


## 5.8 Putting it together — reading `llm_client.py`'s real `_KEYWORD_MAP`

You now have every piece needed to read this for real:

```python
_KEYWORD_MAP = [
    (r"don'?t recognize|didn'?t make this|unauthorized transaction|not my transaction|"
     r"not sure (it|this) was me|wasn'?t me|not sure i made|don'?t think i made|"
     r"(double check|make sure).*(charge|transaction)", "fraud_suspected", 0.9),
    (r"charged twice|double charged|duplicate charge", "payment_issue", 0.9),
    ...
]
```

Each tuple is `(pattern, issue_type, confidence)`. The classifier loops over
this list in order and uses the **first** pattern that matches — which is
exactly why the fraud pattern is listed first: if a ticket matches both a
fraud phrase and, say, the word "refund" somewhere else in the same
sentence, fraud should win.

In [ ]:
import llm_client

# Same two phrasings the project's test suite checks -- one blunt, one hedged.
print(llm_client.mock_classify("T004", "There's a transaction on my account that I don't recognize at all."))
print(llm_client.mock_classify("T005", "Hey, quick one - I noticed a small charge on my account, "
                                        "just wanted to double check, I'm not sure it was me."))


## Exercise

1. Write a pattern that matches `"cancel my order"`, `"cancel the order"`,
   and `"cancel this order"` using a single group with alternation, similar
   to Section 5.4.
2. Extend your pattern with `.*` so it also matches
   `"cancel my order please, it hasn't shipped yet"` (i.e. words between
   "cancel" and "order" shouldn't break the match — think about *where*
   the `.*` needs to go, since "cancel" comes before "order" this time,
   unlike the double-check/charge example).
3. Add a new tuple to a **copy** of `_KEYWORD_MAP` (don't edit the real
   file yet) for an `"order_cancellation"` issue type using your pattern,
   and test it against a few sample tickets.
4. Open `llm_client.py` and find the `_ANGRY_WORDS` and `_FRUSTRATED_WORDS`
   patterns. Write two ticket sentences that should trigger each one, and
   two that should trigger neither.